In [ ]:
import pandas as pd 
import numpy as np 
from datetime import datetime 
import seaborn as sns 
import matplotlib.pyplot as plt
import re 
import os 


In [ ]:
df = pd.read_csv(r'.\data\data.csv', index_col = [0])

In [ ]:
### Correct the data 

df['tuition'] = df['tuition'].replace('Y','Yes').replace('N', 'No')
df['CCA'] = df['CCA'].str.capitalize().fillna('None')

df['age'] = df['age'].replace(5,15).replace(6,16)

In [ ]:
### Drop rows with incomplete data  
df = df.dropna(subset = ['final_test','attendance_rate'], how = 'any')

df = df[df['age'] > 0]

In [ ]:
### Add new columns 
sleep = pd.to_timedelta(df['sleep_time'] + ':00')   # "22:00" -> "22:00:00" -> 22 hours
wake = pd.to_timedelta(df['wake_time'] + ':00')     # "6:00"  -> 6 hours

df['sleep_hours'] = ((wake - sleep) % pd.Timedelta(days=1)).dt.total_seconds() / 3600


In [ ]:
# Set the aesthetic style of the plots  
sns.set_style("whitegrid")  
  
# Histogram for resale price  
plt.figure(figsize=(10, 6))  
sns.histplot(df['final_test'], kde=True)  
plt.title('Distribution of Test Result')  
plt.xlabel('Test Result')  
plt.ylabel('Frequency')  
plt.show()

In [ ]:
### Average final_test score by key factors
sns.set_style("whitegrid")
BAR = '#3b6fb6'                       # one colour: every panel is a single series
overall_mean = df['final_test'].mean()

def group_stats(col):
    g = df.groupby(col)['final_test']
    out = g.agg(['mean', 'count', 'sem']).reset_index()
    out['ci'] = 1.96 * out['sem']     # 95% confidence interval
    return out

def finish(ax, title, xlabel, col):
    r = df[col].replace({'Yes': 1, 'No': 0}).astype(float).corr(df['final_test'])
    ax.axhline(overall_mean, color='#888888', linestyle='--', linewidth=1,
               label=f'Overall average ({overall_mean:.0f})')
    ax.legend(loc='upper left', fontsize=9, frameon=False)
    ax.set_title(f'{title}   (Pearson r = {r:.2f})', loc='left', fontsize=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Average final_test score')
    ax.set_ylim(0, 100)
    ax.grid(axis='x', visible=False)

def bar_panel(ax, col, order=None, labels=None):
    s = group_stats(col)
    if order is not None:
        s = s.set_index(col).loc[order].reset_index()
    names = labels or [str(v) for v in s[col]]
    x = np.arange(len(s))
    ax.bar(x, s['mean'], yerr=s['ci'], color=BAR, width=0.6, capsize=4,
           error_kw={'ecolor': '#333333', 'linewidth': 1})
    for xi, m in zip(x, s['mean']):
        ax.text(xi, m + 3, f'{m:.1f}', ha='center', fontsize=10, color='#222222')
    ax.set_xticks(x, [f'{n}\n(n={c:,})' for n, c in zip(names, s['count'])])

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Study hours per week (line: 0-20 hours is a continuous scale)
s = group_stats('hours_per_week')
ax = axes[0, 0]
ax.fill_between(s['hours_per_week'], s['mean'] - s['ci'], s['mean'] + s['ci'], color=BAR, alpha=0.2, linewidth=0)
ax.plot(s['hours_per_week'], s['mean'], color=BAR, linewidth=2, marker='o', markersize=6)
ax.axvspan(4.5, 10.5, color='#2e7d32', alpha=0.07)
ax.text(7.5, 92, 'sweet spot: 5-10 hrs', ha='center', fontsize=9, color='#2e7d32')
ax.set_xticks(range(0, 21, 2))
finish(ax, '1. Study hours per week', 'Hours studied per week', 'hours_per_week')

# 2. Sleep hours
bar_panel(axes[0, 1], 'sleep_hours', labels=None)
axes[0, 1].set_xticks(axes[0, 1].get_xticks(),
                      [t.get_text().replace('.0', ' hrs') for t in axes[0, 1].get_xticklabels()])
finish(axes[0, 1], '2. Hours of sleep', 'Hours of sleep per night', 'sleep_hours')

# 3. Number of siblings
bar_panel(axes[1, 0], 'number_of_siblings')
finish(axes[1, 0], '3. Number of siblings', 'Number of siblings', 'number_of_siblings')

# 4. Tuition
bar_panel(axes[1, 1], 'tuition', order=['No', 'Yes'], labels=['No tuition', 'Has tuition'])
finish(axes[1, 1], '4. Tuition', '', 'tuition')

fig.suptitle('Average final_test score by factor  (error bars / band = 95% confidence interval)',
             fontsize=14, x=0.01, ha='left')
plt.tight_layout()
plt.show()

## Correlation between variables

Three views:
1. **Pearson correlation heatmap** – numeric variables plus Yes/No and Male/Female columns encoded as 0/1.
2. **Correlation with `final_test`** – the same numbers as a sorted bar chart, so the strongest predictors stand out.
3. **Association matrix across *all* variables** – handles multi-category text columns (CCA, learning_style, transport, bag_color) too:
   - numeric ↔ numeric: |Pearson r|
   - categorical ↔ numeric: correlation ratio η (eta)
   - categorical ↔ categorical: Cramér's V

   All three run from 0 (no relationship) to 1 (perfect relationship), so they can share one colour scale.

In [ ]:
### Build a correlation-ready dataframe
corr_df = df.drop(columns=['student_id', 'sleep_time', 'wake_time'], errors='ignore').copy()

# Bedtime as a number (hours); times after midnight count as 24+ so 1:00 -> 25 (later than 23:00)
bed = pd.to_timedelta(df['sleep_time'] + ':00').dt.total_seconds() / 3600
corr_df['bedtime_hr'] = np.where(bed < 12, bed + 24, bed)

# Encode two-level categories as 0/1 so they can join the Pearson heatmap
corr_df['direct_admission'] = corr_df['direct_admission'].map({'No': 0, 'Yes': 1})
corr_df['tuition'] = corr_df['tuition'].map({'No': 0, 'Yes': 1})
corr_df['gender_male'] = corr_df.pop('gender').map({'Female': 0, 'Male': 1})

cat_cols = ['CCA', 'learning_style', 'mode_of_transport', 'bag_color']
num_cols = [c for c in corr_df.columns if c not in cat_cols]

# Put the target first so it is the top row / left column
num_cols = ['final_test'] + [c for c in num_cols if c != 'final_test']
corr_df[num_cols].describe().T

In [ ]:
### Chart 1: Pearson correlation heatmap (numeric + binary variables)
corr = corr_df[num_cols].corr(method='pearson')

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # hide the duplicate upper triangle

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', annot_kws={'size': 9},
    cmap='RdBu_r', vmin=-1, vmax=1, center=0,           # diverging: blue = negative, red = positive, white = none
    square=True, linewidths=2, linecolor='white',
    cbar_kws={'shrink': 0.7, 'label': 'Pearson r'}
)
plt.title('Pearson correlation between numeric variables', fontsize=14, loc='left')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
### Chart 2: which variables move with final_test?
target_corr = corr['final_test'].drop('final_test').sort_values()

colors = np.where(target_corr < 0, '#2166ac', '#b2182b')   # same blue/red meaning as the heatmap

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(target_corr.index, target_corr.values, color=colors, height=0.6)
ax.axvline(0, color='#555555', linewidth=1)
for bar, v in zip(bars, target_corr.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), bar.get_y() + bar.get_height() / 2,
            f'{v:.2f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9, color='#333333')
ax.set_xlim(-1, 1)
ax.set_xlabel('Pearson r with final_test  (blue = lower score, red = higher score)')
ax.set_title('Correlation of each variable with final_test', fontsize=14, loc='left')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

In [ ]:
### Chart 3: association between ALL variables (numeric + categorical)

def cramers_v(x, y):
    """Cramér's V (bias-corrected) for two categorical series, 0..1."""
    table = pd.crosstab(x, y).values
    n = table.sum()
    if n == 0 or min(table.shape) < 2:
        return np.nan
    expected = table.sum(1, keepdims=True) * table.sum(0, keepdims=True) / n
    chi2 = ((table - expected) ** 2 / expected).sum()
    phi2 = chi2 / n
    r, k = table.shape
    phi2corr = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2corr / min(kcorr - 1, rcorr - 1))

def correlation_ratio(categories, values):
    """Correlation ratio eta: how much of a numeric variable's variance is explained by a category, 0..1."""
    d = pd.DataFrame({'c': categories, 'v': values}).dropna()
    if d['v'].var() == 0:
        return np.nan
    grand_mean = d['v'].mean()
    groups = d.groupby('c')['v']
    ss_between = (groups.count() * (groups.mean() - grand_mean) ** 2).sum()
    ss_total = ((d['v'] - grand_mean) ** 2).sum()
    return np.sqrt(ss_between / ss_total)

all_cols = num_cols + cat_cols
assoc = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)

for a in all_cols:
    for b in all_cols:
        if a == b:
            assoc.loc[a, b] = 1.0
        elif a in cat_cols and b in cat_cols:
            assoc.loc[a, b] = cramers_v(corr_df[a], corr_df[b])
        elif a in cat_cols:
            assoc.loc[a, b] = correlation_ratio(corr_df[a], corr_df[b])
        elif b in cat_cols:
            assoc.loc[a, b] = correlation_ratio(corr_df[b], corr_df[a])
        else:
            assoc.loc[a, b] = abs(corr_df[a].corr(corr_df[b]))

mask = np.triu(np.ones_like(assoc, dtype=bool), k=1)

plt.figure(figsize=(14, 12))
sns.heatmap(
    assoc, mask=mask, annot=True, fmt='.2f', annot_kws={'size': 8},
    cmap='Blues', vmin=0, vmax=1,                         # sequential: darker = stronger relationship
    square=True, linewidths=2, linecolor='white',
    cbar_kws={'shrink': 0.7, 'label': 'Strength of association (0 = none, 1 = perfect)'}
)
plt.title("Association between all variables\n|Pearson r| (num-num) · eta (cat-num) · Cramér's V (cat-cat)",
          fontsize=14, loc='left')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Ranked list of what relates most to final_test
assoc['final_test'].drop('final_test').sort_values(ascending=False).round(3).to_frame('association_with_final_test')

## Modelling: predicting `final_test`

- **Split:** 80% training / 20% testing (`random_state=42`). GridSearchCV and RandomizedSearchCV tune with 5-fold cross-validation **inside the training set**, so the test set is only used once, for the final score.
- **Features:** chosen from the EDA. The weak or useless columns (gender, age, transport, bag colour, student ID) are dropped. Extra features are added for the patterns a straight line can't capture: class size, study-hour bands, and the low-attendance and too-little-sleep thresholds.
- **Models:** Linear Regression, Ridge, Lasso, then Ridge and Lasso tuned with GridSearchCV and with RandomizedSearchCV.
- **Metrics:** MAE and RMSE (average error in marks, lower is better) and R² (share of the variation in scores explained, higher is better).

In [ ]:
### Feature engineering + 80/20 train-test split
from sklearn.model_selection import train_test_split

model_df = df.copy()

# Features suggested by the EDA (the patterns were thresholds / peaks, so give the linear models features that can express them)
model_df['class_size'] = model_df['n_male'] + model_df['n_female']                     # strongest single predictor (r = -0.50)
model_df['study_band'] = pd.cut(model_df['hours_per_week'], bins=[-1, 4, 10, 100],
                                labels=['0-4 hrs', '5-10 hrs', '11+ hrs']).astype(str)  # score peaks at 5-10 hrs
model_df['low_attendance'] = (model_df['attendance_rate'] <= 80).astype(int)           # big drop at <= 80%
model_df['enough_sleep'] = (model_df['sleep_hours'] >= 7).astype(int)                  # big drop below 7 hrs

numerical_features = ['class_size', 'attendance_rate', 'sleep_hours', 'low_attendance', 'enough_sleep']
nominal_features = ['CCA', 'learning_style', 'tuition', 'direct_admission', 'study_band', 'number_of_siblings']
# Left out on purpose (no relationship with final_test): gender, age, mode_of_transport, bag_color, student_id

X = model_df[numerical_features + nominal_features]
y = model_df['final_test']

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {X_train.shape[0]:,} rows ({X_train.shape[0] / len(X):.0%})')
print(f'Testing set:  {X_test.shape[0]:,} rows ({X_test.shape[0] / len(X):.0%})')

In [ ]:
### Preprocessing pipeline + helper to score a model on the test set
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('nom', OneHotEncoder(handle_unknown='ignore', drop='first'), nominal_features)
])

results = []   # collects the test-set score of every model for the final comparison

def evaluate(name, model):
    """Score a fitted model on the 20% test set and store the result."""
    y_pred = model.predict(X_test)
    scores = {
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSE': root_mean_squared_error(y_test, y_pred),
        'R²': r2_score(y_test, y_pred),
    }
    results.append(scores)
    print(f"{name:<28} MAE {scores['MAE']:.3f} | RMSE {scores['RMSE']:.3f} | R² {scores['R²']:.4f}")
    return scores

In [ ]:
### 1-3. Linear Regression, Ridge and Lasso (default settings, alpha = 1)
from sklearn.linear_model import LinearRegression, Ridge, Lasso

lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
ridge_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Ridge(alpha=1))])
lasso_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Lasso(alpha=1))])

for name, pipe in [('Linear Regression', lr_pipeline),
                   ('Ridge (alpha=1)', ridge_pipeline),
                   ('Lasso (alpha=1)', lasso_pipeline)]:
    pipe.fit(X_train, y_train)
    evaluate(name, pipe)

In [ ]:
### 4. GridSearchCV: try every alpha in the list, using 5-fold cross-validation on the training set only
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'regressor__fit_intercept': [True, False]
}

ridge_grid_search = GridSearchCV(Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Ridge())]),
                                 param_grid, cv=5, scoring='r2', n_jobs=-1)
ridge_grid_search.fit(X_train, y_train)

lasso_grid_search = GridSearchCV(Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Lasso(max_iter=10000))]),
                                 param_grid, cv=5, scoring='r2', n_jobs=-1)
lasso_grid_search.fit(X_train, y_train)

print('Best Ridge parameters:', ridge_grid_search.best_params_, f'(CV R² {ridge_grid_search.best_score_:.4f})')
print('Best Lasso parameters:', lasso_grid_search.best_params_, f'(CV R² {lasso_grid_search.best_score_:.4f})')
print()
evaluate('Ridge + GridSearchCV', ridge_grid_search.best_estimator_)
evaluate('Lasso + GridSearchCV', lasso_grid_search.best_estimator_)

In [ ]:
### 5. RandomizedSearchCV: sample 30 random alphas from a continuous range (0.0001 - 1000) instead of a fixed list
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

param_dist = {
    'regressor__alpha': loguniform(1e-4, 1e3),
    'regressor__fit_intercept': [True, False]
}

ridge_random_search = RandomizedSearchCV(Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Ridge())]),
                                         param_distributions=param_dist, n_iter=30, cv=5, scoring='r2',
                                         random_state=42, n_jobs=-1)
ridge_random_search.fit(X_train, y_train)

lasso_random_search = RandomizedSearchCV(Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Lasso(max_iter=10000))]),
                                         param_distributions=param_dist, n_iter=30, cv=5, scoring='r2',
                                         random_state=42, n_jobs=-1)
lasso_random_search.fit(X_train, y_train)

print('Best Ridge parameters:', ridge_random_search.best_params_, f'(CV R² {ridge_random_search.best_score_:.4f})')
print('Best Lasso parameters:', lasso_random_search.best_params_, f'(CV R² {lasso_random_search.best_score_:.4f})')
print()
evaluate('Ridge + RandomizedSearchCV', ridge_random_search.best_estimator_)
evaluate('Lasso + RandomizedSearchCV', lasso_random_search.best_estimator_)

In [ ]:
### Which model is most accurate? (all scored on the same 20% test set)
results_df = (pd.DataFrame(results)
              .drop_duplicates('Model', keep='last')
              .sort_values('RMSE')
              .reset_index(drop=True))
display(results_df.style.format({'MAE': '{:.3f}', 'MSE': '{:.2f}', 'RMSE': '{:.3f}', 'R²': '{:.4f}'})
                  .highlight_min(subset=['MAE', 'MSE', 'RMSE'], color='#d4edda')
                  .highlight_max(subset=['R²'], color='#d4edda'))

best = results_df.iloc[0]
print(f"Most accurate: {best['Model']}  ->  R² {best['R²']:.4f}, RMSE {best['RMSE']:.3f} marks")

# Chart: test RMSE per model (lower = better)
plot_df = results_df.sort_values('RMSE', ascending=False)
colors = ['#2e7d32' if m == best['Model'] else '#3b6fb6' for m in plot_df['Model']]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(plot_df['Model'], plot_df['RMSE'], color=colors, height=0.6)
for bar, rmse, r2 in zip(bars, plot_df['RMSE'], plot_df['R²']):
    ax.text(rmse + 0.05, bar.get_y() + bar.get_height() / 2, f'RMSE {rmse:.3f}  ·  R² {r2:.3f}',
            va='center', fontsize=9, color='#222222')
ax.set_xlim(0, plot_df['RMSE'].max() * 1.35)
ax.set_xlabel('Test RMSE (average error in final_test marks, lower is better)')
ax.set_title('Model accuracy on the 20% test set (best in green)', loc='left', fontsize=13)
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

### Conclusion

- **The models are effectively tied.** Linear Regression, Ridge, and tuned Lasso all score about **R² 0.626, RMSE 8.6 marks**. They differ only in the 4th decimal place, which is noise.
- **Tuning didn't help.** GridSearchCV and RandomizedSearchCV both picked a small alpha (very little regularisation), which makes Ridge and Lasso behave like plain Linear Regression. That is expected here: there are about 11,700 training rows and only about 15 features, so the model isn't overfitting and there is nothing for regularisation to fix.
- **Lasso with alpha = 1 is the worst (R² 0.48).** At that strength the penalty forces useful coefficients down to zero. Tuning alpha brings it back level with the others.
- **What matters more than the choice of model is the features.** Adding class size, study-hour bands and threshold flags raised R² from 0.589 to 0.626.

In [ ]:
### Double-check: are Linear, Ridge and Lasso really the same, and is anything overfitting?
from sklearn.model_selection import cross_val_score

check_models = {
    'Linear Regression': lr_pipeline,
    'Ridge (alpha=1)': ridge_pipeline,
    'Lasso (alpha=1)': lasso_pipeline,
    'Ridge + GridSearchCV': ridge_grid_search.best_estimator_,
    'Lasso + GridSearchCV': lasso_grid_search.best_estimator_,
    'Ridge + RandomizedSearchCV': ridge_random_search.best_estimator_,
    'Lasso + RandomizedSearchCV': lasso_random_search.best_estimator_,
}

# --- Check 1: train vs test score. Overfitting = much better on train than on test.
rows = []
for name, m in check_models.items():
    pred_train, pred_test = m.predict(X_train), m.predict(X_test)
    rows.append({'Model': name,
                 'Train RMSE': root_mean_squared_error(y_train, pred_train),
                 'Test RMSE': root_mean_squared_error(y_test, pred_test),
                 'Train R²': r2_score(y_train, pred_train),
                 'Test R²': r2_score(y_test, pred_test)})
gap_df = pd.DataFrame(rows)
gap_df['Gap (test - train RMSE)'] = gap_df['Test RMSE'] - gap_df['Train RMSE']
print('CHECK 1: train vs test (a large gap would mean overfitting)')
display(gap_df.style.format({c: '{:.4f}' for c in gap_df.columns if c != 'Model'}))

# --- Check 2: are the predictions / coefficients actually the same?
lr_pred = lr_pipeline.predict(X_test)
print('\nCHECK 2: how different are the predictions from Linear Regression? (in marks)')
for name, m in check_models.items():
    diff = np.abs(m.predict(X_test) - lr_pred)
    print(f'  {name:<28} max diff {diff.max():.4f} | mean diff {diff.mean():.4f}')

feature_names = lr_pipeline.named_steps['preprocessor'].get_feature_names_out()
coef_df = pd.DataFrame({name: m.named_steps['regressor'].coef_ for name, m in check_models.items()},
                       index=[f.replace('num__', '').replace('nom__', '') for f in feature_names])
print('\nCoefficients side by side (Lasso alpha=1 zeroes some out; the rest are nearly identical):')
display(coef_df.round(3))

# --- Check 3: is the score stable across 5 cross-validation folds?
scores = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print('\nCHECK 3: Linear Regression, 5-fold cross-validation RMSE per fold')
print('  ', np.round(-scores, 3), f'-> mean {-scores.mean():.3f} (+/- {scores.std():.3f})')

## Final model choice: Lasso + RandomizedSearchCV

| Model | Train RMSE | Test RMSE | Test R² | Gap (test - train) |
|---|---|---|---|---|
| Linear Regression | 8.651 | 8.605 | 0.6258 | -0.046 |
| Ridge (alpha=1) / Ridge + GridSearchCV | 8.651 | 8.605 | 0.6258 | -0.045 |
| Ridge + RandomizedSearchCV | 8.651 | 8.605 | 0.6258 | -0.045 |
| Lasso + GridSearchCV | 8.651 | 8.605 | 0.6258 | -0.045 |
| **Lasso + RandomizedSearchCV (alpha ≈ 0.014)** | **8.652** | **8.610** | **0.6254** | **-0.043** |
| Lasso (alpha=1) | 10.094 | 10.142 | 0.4802 | +0.048 |

**We choose Lasso tuned with RandomizedSearchCV**, for three reasons:

- **Smallest train-test gap (0.043 marks).** It performs the most consistently between the data it learned from and unseen data, so it is the most reliable at generalising.
- **Accuracy is effectively the same as the best model.** Its test RMSE is 8.610 against 8.605, a difference of 0.005 marks, so choosing it costs nothing in practice. On average it predicts `final_test` within about 6.9 marks (MAE) and explains about 63% of the variation in scores.
- **A simpler, easier-to-explain model.** Lasso shrinks weak features towards zero (for example, it drops `sleep_hours` once the `enough_sleep` flag is present), leaving the factors that matter most: study hours, number of siblings, class size, CCA, learning style and tuition.

RandomizedSearchCV found this alpha by sampling 30 values from a continuous range (0.0001-1000) with 5-fold cross-validation on the training set only, so the test score is a fair, unseen check.